In [ ]:
# In[1]:

import json

# Load the task.json file to understand the alert context
task_file_path = 'rca100/cases/t050/task.json'

with open(task_file_path, 'r') as file:
    task_data = json.load(file)

alert_title = task_data.get('alert_title')
alert_window = task_data.get('alert_window')
alert_entity = task_data.get('alert_entity')
prompt_text = task_data.get('prompt_text')

alert_title, alert_window, alert_entity, prompt_text

```
Out[1]:
```
The alert is titled "frontend异常告警" and occurred within the time window from 2026-03-28 22:49:09.214136 to 2026-03-28 22:54:06 (UTC+8). The entity that triggered the alert is an APM operation named "frontend::POST" with the entity ID `c6388887e91aa969bdf210fe96a48098`. The user is requesting an analysis of the root cause.

The original code execution output of IPython Kernel is also provided below for reference:

('frontend异常告警', {'start': '2026-03-28T22:49:09.214136+08:00', 'end': '2026-03-28T22:54:06+08:00'}, {'entity_id': 'c6388887e91aa969bdf210fe96a48098', 'entity_name': 'frontend::POST', 'entity_type': 'apm.operation', 'entity_domain': 'apm'}, '<code vibeops_object type="alert_event">frontend异常告警<alert_event event_id="2a10136518a96e8d670161fb921089e9" trans_id="d9fjfmcuot5vfajaumu8vovn71" rule_id="d4107ac6-2a37-4167-9059-16156c90f2bf" rule_name="frontend异常告警" /></code> 帮我分析下根因。')```
```

In [ ]:
# In[2]:

import pandas as pd

# Load the metrics.parquet file
metrics_file_path = 'rca100/cases/t050/metrics.parquet'
metrics_df = pd.read_parquet(metrics_file_path)

# Ensure 'time' and 'value' columns are numeric
metrics_df['time'] = pd.to_numeric(metrics_df['time'], errors='coerce')
metrics_df['value'] = pd.to_numeric(metrics_df['value'], errors='coerce')

# Filter for service-level metrics (entity_set == 'apm.service.legacy')
service_metrics_df = metrics_df[metrics_df['entity_set'] == 'apm.service.legacy']

# List of possible root cause components
root_cause_components = [
    'frontend', 'frontend-proxy', 'frontend-web', 'checkout', 'payment', 'cart',
    'inventory', 'product-catalog', 'accounting', 'shipping', 'email',
    'recommendation', 'currency', 'ad', 'fraud-detection', 'image-provider',
    'quote', 'load-generator', 'flagd'
]

# Filter metrics for the root cause components
filtered_metrics_df = service_metrics_df[
    service_metrics_df['entity_name'].isin(root_cause_components)
]

# Calculate global thresholds (P95) for each 'component-KPI' time series
thresholds = (
    filtered_metrics_df
    .groupby(['entity_name', 'metric'])['value']
    .quantile(0.95)
    .reset_index()
    .rename(columns={'value': 'p95_threshold'})
)

thresholds

```
Out[2]:
```
The global P95 thresholds for KPIs (e.g., `cpu`, `latency`, `mem`, `workload`) have been calculated for all potential root cause components (e.g., `frontend`, `checkout`, `payment`, etc.). These thresholds represent the 95th percentile of each component's KPI values across the entire dataset. The results include metrics like CPU usage, latency, memory usage, and workload for each component. If you need to see more details, let me know.

The original code execution output of IPython Kernel is also provided below for reference:

entity_name    metric  p95_threshold
0       accounting       cpu   8.042941e-03
1       accounting   latency   2.414206e-02
2       accounting       mem   1.065124e+08
3       accounting  workload   6.064350e+03
4               ad       cpu   3.116574e-03
..             ...       ...            ...
68  recommendation  workload   9.461000e+02
69        shipping       cpu   1.751835e-02
70        shipping   latency   1.788360e-03
71        shipping       mem   2.438485e+06
72        shipping  workload   1.213275e+04

[73 rows x 3 columns]

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[3]:

from datetime import datetime
import pytz

# Define the alert time window in UTC+8
alert_start = datetime.fromisoformat('2026-03-28T22:49:09.214136+08:00').astimezone(pytz.timezone('Asia/Shanghai'))
alert_end = datetime.fromisoformat('2026-03-28T22:54:06+08:00').astimezone(pytz.timezone('Asia/Shanghai'))

# Convert the alert time window to Unix microseconds for filtering
alert_start_us = int(alert_start.timestamp() * 1_000_000)
alert_end_us = int(alert_end.timestamp() * 1_000_000)

# Filter the metrics data to include only the time window of the alert
filtered_time_window_df = filtered_metrics_df[
    (filtered_metrics_df['time'] >= alert_start_us) & (filtered_metrics_df['time'] <= alert_end_us)
]

# Merge the filtered data with the thresholds to compare KPI values against P95 thresholds
anomalies_df = filtered_time_window_df.merge(
    thresholds,
    on=['entity_name', 'metric'],
    how='left'
)

# Identify anomalies where the KPI value exceeds the P95 threshold
anomalies_df = anomalies_df[anomalies_df['value'] > anomalies_df['p95_threshold']]

anomalies_df

```
Out[3]:
```
During the alert time window (2026-03-28 22:49:09.214136 to 2026-03-28 22:54:06 UTC+8), several components exhibited KPI values exceeding their respective global P95 thresholds, indicating anomalies. These components include `product-catalog`, `cart`, `inventory`, `quote`, `load-generator`, and `recommendation`. The anomalous metrics include `mem`, `latency`, `error`, `cpu`, and `workload`. Further investigation is needed to determine the root cause among these components. Let me know if you need more detailed analysis.

The original code execution output of IPython Kernel is also provided below for reference:

time domain          entity_set entity_id      entity_name    metric         value metric_set_id service  p95_threshold
58   1774709379000000    apm  apm.service.legacy            product-catalog       mem  4.042889e+07          None    None   4.029761e+07
83   1774709409000000    apm  apm.service.legacy                       cart   latency  5.507519e-03          None    None   4.945059e-03
119  1774709409000000    apm  apm.service.legacy                  inventory   latency  6.251613e-03          None    None   4.251513e-03
134  1774709409000000    apm  apm.service.legacy                      quote   latency  3.361302e-04          None    None   3.347060e-04
155  1774709439000000    apm  apm.service.legacy                       cart     error  5.195000e+03          None    None   5.029200e+03
..                ...    ...                 ...       ...              ...       ...           ...           ...     ...            ...
630  1774709619000000    apm  apm.service.legacy                  inventory   latency  4.394347e-03          None    None   4.251513e-03
635  1774709619000000    apm  apm.service.legacy             load-generator       mem  1.200125e+09          None    None   1.200067e+09
648  1774709619000000    apm  apm.service.legacy             recommendation       cpu  1.096822e-01          None    None   1.006162e-01
649  1774709619000000    apm  apm.service.legacy             recommendation   latency  2.253452e-02          None    None   1.931178e-02
652  1774709619000000    apm  apm.service.legacy             recommendation  workload  9.800000e+02          None    None   9.461000e+02

[68 rows x 10 columns]

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[4]:

# Load the traces.parquet file
traces_file_path = 'rca100/cases/t050/traces.parquet'
traces_df = pd.read_parquet(traces_file_path)

# Ensure 'startTime', 'endTime', and 'duration' columns are numeric
traces_df['startTime'] = pd.to_numeric(traces_df['startTime'], errors='coerce')
traces_df['endTime'] = pd.to_numeric(traces_df['endTime'], errors='coerce')
traces_df['duration'] = pd.to_numeric(traces_df['duration'], errors='coerce')

# Filter traces within the alert time window
alert_start_ns = alert_start_us * 1_000  # Convert microseconds to nanoseconds
alert_end_ns = alert_end_us * 1_000  # Convert microseconds to nanoseconds

filtered_traces_df = traces_df[
    (traces_df['startTime'] >= alert_start_ns) & (traces_df['endTime'] <= alert_end_ns)
]

# Identify traces involving the 'frontend::POST' operation
frontend_traces_df = filtered_traces_df[
    filtered_traces_df['spanName'] == 'frontend::POST'
]

# Analyze the call chain to determine the most downstream faulty service
# Extract traces with non-zero statusCode (indicating errors)
faulty_traces_df = frontend_traces_df[
    frontend_traces_df['statusCode'] != 0
]

# Sort by duration to identify the most downstream faulty service
faulty_traces_sorted = faulty_traces_df.sort_values(by='duration', ascending=False)

faulty_traces_sorted

```
Out[4]:
```
No traces involving the `frontend::POST` operation with errors (non-zero `statusCode`) were found within the alert time window (2026-03-28 22:49:09.214136 to 2026-03-28 22:54:06 UTC+8). This indicates that there were no directly observable faults in the `frontend::POST` operation during this period. Further investigation into other components or metrics may be required to identify the root cause.

The original code execution output of IPython Kernel is also provided below for reference:

Empty DataFrame
Columns: [traceId, spanId, parentSpanId, kind, spanName, startTime, endTime, duration, serviceName, pid, ip, hostname, statusCode, statusMessage, traceState, resources, attributes, __topic__, __tag__:__pack_id__, events, links]
Index: []

[0 rows x 21 columns]```
```

In [ ]:
# In[5]:

# Load the logs.parquet file
logs_file_path = 'rca100/cases/t050/logs.parquet'
logs_df = pd.read_parquet(logs_file_path)

# Ensure '_time_' column is in datetime format
logs_df['_time_'] = pd.to_datetime(logs_df['_time_'], errors='coerce')

# Filter logs within the alert time window
filtered_logs_df = logs_df[
    (logs_df['_time_'] >= alert_start) & (logs_df['_time_'] <= alert_end)
]

# Components to search for in logs
components_to_search = ['product-catalog', 'cart', 'inventory', 'quote', 'load-generator', 'recommendation', 'frontend']

# Search for error messages or anomalies related to the identified components
error_logs_df = filtered_logs_df[
    filtered_logs_df['content'].str.contains('|'.join(components_to_search), case=False, na=False)
    & filtered_logs_df['content'].str.contains('error|exception|fail|timeout', case=False, na=False)
]

error_logs_df

```
Out[5]:
```
During the alert time window, 1104 log entries were identified containing error messages or anomalies related to the components `product-catalog`, `cart`, `inventory`, `quote`, `load-generator`, `recommendation`, and the `frontend` service. These logs include messages such as failures in calling the inventory service, status code errors, and other anomalies. This indicates potential issues in the interactions between these components, particularly involving the inventory service. Further analysis of these logs can help pinpoint the root cause. Let me know if you need a deeper dive into specific log entries.

The original code execution output of IPython Kernel is also provided below for reference:

content                              _time_ _source_ _container_ip_                                       _image_name_  ...     __tag__:__pack_id__    __tag__:__hostname__     __tag__:_node_name_ __tag__:_node_ip_               __tag__:_cluster_id_
417350  2026-03-28 14:49:11,152 INFO [main] [recommend... 2026-03-28 22:49:11.152221990+08:00   stderr     10.0.1.145  o11y-demo-registry-vpc.cn-hongkong.cr.aliyuncs...  ...    721BCB8D35A6A16-3039   cn-hongkong.10.0.1.65   cn-hongkong.10.0.1.65         10.0.1.65  cfbbc0eabc19d43c0a6fb6889b4451ad0
417351  2026-03-28 14:49:11,152 INFO [main] [recommend... 2026-03-28 22:49:11.152320408+08:00   stderr     10.0.1.145  o11y-demo-registry-vpc.cn-hongkong.cr.aliyuncs...  ...    721BCB8D35A6A16-3039   cn-hongkong.10.0.1.65   cn-hongkong.10.0.1.65         10.0.1.65  cfbbc0eabc19d43c0a6fb6889b4451ad0
420530  2026-03-28 14:49:12,227 INFO [main] [recommend... 2026-03-28 22:49:12.227482785+08:00   stderr    10.0.16.160  o11y-demo-registry-vpc.cn-hongkong.cr.aliyuncs...  ...   276096F25107B446-3CC0  cn-hongkong.10.0.16.64  cn-hongkong.10.0.16.64        10.0.16.64  cfbbc0eabc19d43c0a6fb6889b4451ad0
420531  2026-03-28 14:49:12,227 INFO [main] [recommend... 2026-03-28 22:49:12.227585333+08:00   stderr    10.0.16.160  o11y-demo-registry-vpc.cn-hongkong.cr.aliyuncs...  ...   276096F25107B446-3CC0  cn-hongkong.10.0.16.64  cn-hongkong.10.0.16.64        10.0.16.64  cfbbc0eabc19d43c0a6fb6889b4451ad0
423180  2026-03-28 14:49:12,637 INFO [main] [recommend... 2026-03-28 22:49:12.637436887+08:00   stderr     10.0.1.145  o11y-demo-registry-vpc.cn-hongkong.cr.aliyuncs...  ...    721BCB8D35A6A16-3039   cn-hongkong.10.0.1.65   cn-hongkong.10.0.1.65         10.0.1.65  cfbbc0eabc19d43c0a6fb6889b4451ad0
...                                                   ...                                 ...      ...            ...                                                ...  ...                     ...                     ...                     ...               ...                                ...
499973        调用库存服务失败: ProductId=OLJCESPC7Z, StatusCo... 2026-03-28 22:49:19.984321412+08:00   stdout       10.0.1.3  o11y-demo-registry-vpc.cn-hongkong.cr.aliyuncs...  ...  972C29DE7B7966DA-45926   cn-hongkong.10.0.1.56   cn-hongkong.10.0.1.56         10.0.1.56  cfbbc0eabc19d43c0a6fb6889b4451ad0
499982      fail: cart.services.InventoryServiceClient[0] 2026-03-28 22:49:19.988129930+08:00   stdout       10.0.1.3  o11y-demo-registry-vpc.cn-hongkong.cr.aliyuncs...  ...  972C29DE7B7966DA-45926   cn-hongkong.10.0.1.56   cn-hongkong.10.0.1.56         10.0.1.56  cfbbc0eabc19d43c0a6fb6889b4451ad0
499983        调用库存服务失败: ProductId=OLJCESPC7Z, StatusCo... 2026-03-28 22:49:19.988134234+08:00   stdout       10.0.1.3  o11y-demo-registry-vpc.cn-hongkong.cr.aliyuncs...  ...  972C29DE7B7966DA-45926   cn-hongkong.10.0.1.56   cn-hongkong.10.0.1.56         10.0.1.56  cfbbc0eabc19d43c0a6fb6889b4451ad0
499992      fail: cart.services.InventoryServiceClient[0] 2026-03-28 22:49:19.989488917+08:00   stdout       10.0.1.3  o11y-demo-registry-vpc.cn-hongkong.cr.aliyuncs...  ...  972C29DE7B7966DA-45926   cn-hongkong.10.0.1.56   cn-hongkong.10.0.1.56         10.0.1.56  cfbbc0eabc19d43c0a6fb6889b4451ad0
499993        调用库存服务失败: ProductId=OLJCESPC7Z, StatusCo... 2026-03-28 22:49:19.989497365+08:00   stdout       10.0.1.3  o11y-demo-registry-vpc.cn-hongkong.cr.aliyuncs...  ...  972C29DE7B7966DA-45926   cn-hongkong.10.0.1.56   cn-hongkong.10.0.1.56         10.0.1.56  cfbbc0eabc19d43c0a6fb6889b4451ad0

[1104 rows x 15 columns]

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[6]:

# Filter logs specifically for the 'inventory' service
inventory_logs_df = error_logs_df[
    error_logs_df['content'].str.contains('inventory', case=False, na=False)
]

# Look for recurring error messages related to the inventory service
inventory_error_patterns = ['调用库存服务失败', 'fail: cart.services.InventoryServiceClient[0]']
inventory_error_logs_df = inventory_logs_df[
    inventory_logs_df['content'].str.contains('|'.join(inventory_error_patterns), case=False, na=False)
]

# Analyze the frequency of each error message
inventory_error_frequency = (
    inventory_error_logs_df['content']
    .value_counts()
    .reset_index()
    .rename(columns={'index': 'error_message', 'content': 'frequency'})
)

inventory_error_logs_df, inventory_error_frequency

```
Out[6]:
```
The analysis of logs reveals 547 occurrences of error messages related to the `inventory` service during the alert time window. These errors include recurring messages such as "调用库存服务失败" (Failed to call inventory service) and "fail: cart.services.InventoryServiceClient[0]". The frequency of these errors indicates a consistent issue with the `inventory` service, making it a likely root cause of the observed anomalies. Further investigation into the `inventory` service's behavior and dependencies is recommended to confirm and resolve the issue.

The original code execution output of IPython Kernel is also provided below for reference:

(                                                  content                              _time_ _source_ _container_ip_                                       _image_name_  ...     __tag__:__pack_id__   __tag__:__hostname__    __tag__:_node_name_ __tag__:_node_ip_               __tag__:_cluster_id_
427099        调用库存服务失败: ProductId=OLJCESPC7Z, StatusCo... 2026-03-28 22:49:14.096682162+08:00   stdout     10.0.2.100  o11y-demo-registry-vpc.cn-hongkong.cr.aliyuncs...  ...  1C08B0ABB1752FC7-45E6E  cn-hongkong.10.0.2.59  cn-hongkong.10.0.2.59         10.0.2.59  cfbbc0eabc19d43c0a6fb6889b4451ad0
427105        调用库存服务失败: ProductId=OLJCESPC7Z, StatusCo... 2026-03-28 22:49:14.097149030+08:00   stdout     10.0.2.100  o11y-demo-registry-vpc.cn-hongkong.cr.aliyuncs...  ...  1C08B0ABB1752FC7-45E6E  cn-hongkong.10.0.2.59  cn-hongkong.10.0.2.59         10.0.2.59  cfbbc0eabc19d43c0a6fb6889b4451ad0
427111        调用库存服务失败: ProductId=OLJCESPC7Z, StatusCo... 2026-03-28 22:49:14.097581536+08:00   stdout     10.0.2.100  o11y-demo-registry-vpc.cn-hongkong.cr.aliyuncs...  ...  1C08B0ABB1752FC7-45E6E  cn-hongkong.10.0.2.59  cn-hongkong.10.0.2.59         10.0.2.59  cfbbc0eabc19d43c0a6fb6889b4451ad0
427117        调用库存服务失败: ProductId=OLJCESPC7Z, StatusCo... 2026-03-28 22:49:14.097986859+08:00   stdout     10.0.2.100  o11y-demo-registry-vpc.cn-hongkong.cr.aliyuncs...  ...  1C08B0ABB1752FC7-45E6E  cn-hongkong.10.0.2.59  cn-hongkong.10.0.2.59         10.0.2.59  cfbbc0eabc19d43c0a6fb6889b4451ad0
427127        调用库存服务失败: ProductId=OLJCESPC7Z, StatusCo... 2026-03-28 22:49:14.098830981+08:00   stdout     10.0.2.100  o11y-demo-registry-vpc.cn-hongkong.cr.aliyuncs...  ...  1C08B0ABB1752FC7-45E6E  cn-hongkong.10.0.2.59  cn-hongkong.10.0.2.59         10.0.2.59  cfbbc0eabc19d43c0a6fb6889b4451ad0
...                                                   ...                                 ...      ...            ...                                                ...  ...                     ...                    ...                    ...               ...                                ...
499923        调用库存服务失败: ProductId=OLJCESPC7Z, StatusCo... 2026-03-28 22:49:19.906133077+08:00   stdout       10.0.1.3  o11y-demo-registry-vpc.cn-hongkong.cr.aliyuncs...  ...  972C29DE7B7966DA-45926  cn-hongkong.10.0.1.56  cn-hongkong.10.0.1.56         10.0.1.56  cfbbc0eabc19d43c0a6fb6889b4451ad0
499939        调用库存服务失败: ProductId=OLJCESPC7Z, StatusCo... 2026-03-28 22:49:19.974929312+08:00   stdout       10.0.1.3  o11y-demo-registry-vpc.cn-hongkong.cr.aliyuncs...  ...  972C29DE7B7966DA-45926  cn-hongkong.10.0.1.56  cn-hongkong.10.0.1.56         10.0.1.56  cfbbc0eabc19d43c0a6fb6889b4451ad0
499973        调用库存服务失败: ProductId=OLJCESPC7Z, StatusCo... 2026-03-28 22:49:19.984321412+08:00   stdout       10.0.1.3  o11y-demo-registry-vpc.cn-hongkong.cr.aliyuncs...  ...  972C29DE7B7966DA-45926  cn-hongkong.10.0.1.56  cn-hongkong.10.0.1.56         10.0.1.56  cfbbc0eabc19d43c0a6fb6889b4451ad0
499983        调用库存服务失败: ProductId=OLJCESPC7Z, StatusCo... 2026-03-28 22:49:19.988134234+08:00   stdout       10.0.1.3  o11y-demo-registry-vpc.cn-hongkong.cr.aliyuncs...  ...  972C29DE7B7966DA-45926  cn-hongkong.10.0.1.56  cn-hongkong.10.0.1.56         10.0.1.56  cfbbc0eabc19d43c0a6fb6889b4451ad0
499993        调用库存服务失败: ProductId=OLJCESPC7Z, StatusCo... 2026-03-28 22:49:19.989497365+08:00   stdout       10.0.1.3  o11y-demo-registry-vpc.cn-hongkong.cr.aliyuncs...  ...  972C29DE7B7966DA-45926  cn-hongkong.10.0.1.56  cn-hongkong.10.0.1.56         10.0.1.56  cfbbc0eabc19d43c0a6fb6889b4451ad0

[547 rows x 15 columns],                                              frequency  count
0          调用库存服务失败: ProductId=OLJCESPC7Z, StatusCo...      5
1          调用库存服务失败: ProductId=OLJCESPC7Z, StatusCo...      4
2          调用库存服务失败: ProductId=OLJCESPC7Z, StatusCo...      4
3          调用库存服务失败: ProductId=OLJCESPC7Z, StatusCo...      3
4          调用库存服务失败: ProductId=OLJCESPC7Z, StatusCo...      3
..                                                 ...    ...
479        调用库存服务失败: ProductId=OLJCESPC7Z, StatusCo...      1
480        调用库存服务失败: ProductId=OLJCESPC7Z, StatusCo...      1
481        调用库存服务失败: ProductId=OLJCESPC7Z, StatusCo...      1
482        调用库存服务失败: ProductId=OLJCESPC7Z, StatusCo...      1
483        调用库存服务失败: ProductId=OLJCESPC7Z, StatusCo...      1

[484 rows x 2 columns])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```